In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:
DATA_DIR = r"C:\Users\Admin\Desktop\CropGuard\CropGuard\ml\data\plantvillage_segmented"

IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.3,
        hue=0.05
    ),
    transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))],
        p=0.3
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    transforms.RandomErasing(
        p=0.2,
        scale=(0.02, 0.1)
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Create dataset without permanently attaching train augmentation
full_dataset = datasets.ImageFolder(DATA_DIR)

num_classes = len(full_dataset.classes)

print(f"Found {num_classes} classes")
print(full_dataset.classes)

# Split indices
val_size = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_indices, val_indices = random_split(
    range(len(full_dataset)),
    [train_size, val_size]
)

# Create separate datasets so train and validation
# can use different transforms
train_dataset = datasets.ImageFolder(
    DATA_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    DATA_DIR,
    transform=val_transform
)

# Apply the same split indices
train_dataset = torch.utils.data.Subset(
    train_dataset,
    train_indices.indices
)

val_dataset = torch.utils.data.Subset(
    val_dataset,
    val_indices.indices
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

print(f"Train size: {train_size}, Val size: {val_size}")

Found 38 classes
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'T

In [3]:
print(train_loader)
print(val_loader)
print(f"Train size: {train_size}, Val size: {val_size}")

Train size: 48875, Val size: 5430


In [4]:
# Properly separate train/val transforms (random_split shares the underlying dataset object)
val_dataset.dataset = datasets.ImageFolder(DATA_DIR, transform=val_transform)

# Load pretrained MobileNetV2
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze all feature layers — we only train the classifier head
for param in model.features.parameters():
    param.requires_grad = False

# Swap the final classifier layer for our 38 classes
model.classifier[1] = nn.Linear(model.last_channel, num_classes)

model = model.to(device)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=38, bias=True)
)


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

EPOCHS = 15

def train_one_epoch():
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate():
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

best_val_acc = 0.0
SAVE_PATH = r"C:\Users\Admin\Desktop\CropGuard\CropGuard\ml\models\cropguard_seg_aug.pt"

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = validate()
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': full_dataset.classes,
            'val_acc': val_acc,
        }, SAVE_PATH)
        print(f"  → New best model saved (val_acc: {val_acc:.2f}%)")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.2f}%")

Epoch 1/15 | Train Loss: 0.9747 Acc: 75.27% | Val Loss: 0.3786 Acc: 89.13%
  → New best model saved (val_acc: 89.13%)
Epoch 2/15 | Train Loss: 0.5435 Acc: 83.94% | Val Loss: 0.2910 Acc: 90.55%
  → New best model saved (val_acc: 90.55%)
Epoch 3/15 | Train Loss: 0.4859 Acc: 85.14% | Val Loss: 0.2583 Acc: 91.38%
  → New best model saved (val_acc: 91.38%)
Epoch 4/15 | Train Loss: 0.4564 Acc: 85.79% | Val Loss: 0.2178 Acc: 92.85%
  → New best model saved (val_acc: 92.85%)
Epoch 5/15 | Train Loss: 0.4360 Acc: 86.12% | Val Loss: 0.2116 Acc: 92.87%
  → New best model saved (val_acc: 92.87%)
Epoch 6/15 | Train Loss: 0.4309 Acc: 86.08% | Val Loss: 0.2046 Acc: 93.30%
  → New best model saved (val_acc: 93.30%)
Epoch 7/15 | Train Loss: 0.4207 Acc: 86.48% | Val Loss: 0.2016 Acc: 93.13%
Epoch 8/15 | Train Loss: 0.4111 Acc: 86.70% | Val Loss: 0.2034 Acc: 92.82%
Epoch 9/15 | Train Loss: 0.4135 Acc: 86.73% | Val Loss: 0.2022 Acc: 93.15%
Epoch 10/15 | Train Loss: 0.4074 Acc: 86.85% | Val Loss: 0.1959 Acc